In [7]:
import copy
import hashlib
import random
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from mordred import Calculator
from mordred import (
    ABCIndex,
    AcidBase,
    Aromatic,
    BCUT,
    BondCount,
    CarbonTypes,
    Chi,
    Constitutional,
    CPSA,
    EState,
    EccentricConnectivityIndex,
    HydrogenBond,
    KappaShapeIndex,
    Lipinski,
    LogS,
    McGowanVolume,
    MoeType,
    Polarizability,
    RingCount,
    RotatableBond,
    SLogP,
    TopoPSA,
    TopologicalCharge,
    TopologicalIndex,
    VdwVolumeABC,
    Weight,
    ZagrebIndex,
)
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, TensorDataset


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

DESCRIPTOR_CLASSES = [
    SLogP,
    TopoPSA,
    HydrogenBond,
    McGowanVolume,
    Polarizability,
    Weight,
    LogS,
    Lipinski,
    MoeType,
    CPSA,
    RotatableBond,
    RingCount,
    Aromatic,
    KappaShapeIndex,
    Constitutional,
    EState,
    BCUT,
    VdwVolumeABC,
    TopologicalIndex,
    ZagrebIndex,
    ABCIndex,
    AcidBase,
    BondCount,
    CarbonTypes,
    Chi,
    EccentricConnectivityIndex,
    TopologicalCharge,
]


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_calculator():
    return Calculator(DESCRIPTOR_CLASSES, ignore_3D=True)


def smiles_to_mordred(smiles, calculator):
    """Return finite descriptors; Mordred failures are represented as NaN."""
    n_desc = len(calculator.descriptors)
    if pd.isna(smiles) or not str(smiles).strip():
        return np.full(n_desc, np.nan, dtype=np.float64)
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.full(n_desc, np.nan, dtype=np.float64)

    values = []
    for value in calculator(mol):
        try:
            value = float(value)
        except (TypeError, ValueError, OverflowError):
            value = np.nan
        values.append(value if np.isfinite(value) else np.nan)
    return np.asarray(values, dtype=np.float64)

class TrainOnlyStandardizer:
    """Drop columns with any NaN in training data, remove train-constant columns, then standardize."""

    def fit(self, arrays):
        stacked = np.vstack(arrays).astype(np.float64)
        finite = np.isfinite(stacked)

        # Drop any column that has at least one non-finite (NaN/inf) value in training data
        no_missing = finite.all(axis=0)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            self.mean = np.nanmean(stacked, axis=0)
            self.scale = np.nanstd(stacked, axis=0)

        non_constant = self.scale > 1e-12
        self.keep = no_missing & non_constant

        self.mean = np.where(self.keep, self.mean, 0.0)
        self.scale = np.where(self.keep, self.scale, 1.0)

        n_total = stacked.shape[1]
        n_nan_dropped = int((~no_missing).sum())
        n_constant_dropped = int((no_missing & ~non_constant).sum())
        n_remaining = int(self.keep.sum())
        print(
            f"[TrainOnlyStandardizer] Descriptors: {n_total} total | "
            f"{n_nan_dropped} dropped (NaN) | "
            f"{n_constant_dropped} dropped (constant) | "
            f"{n_remaining} remaining"
        )

        return self

    def transform(self, array):
        array = np.asarray(array, dtype=np.float64)
        filled = np.where(np.isfinite(array), array, self.mean)
        return ((filled - self.mean) / self.scale)[:, self.keep]

    @property
    def n_features_out(self):
        return int(self.keep.sum())


def get_canonical_smiles(smiles_str):
    """Convert SMILES string to canonical format for reliable matching."""
    try:
        mol = Chem.MolFromSmiles(str(smiles_str))
        return Chem.MolToSmiles(mol) if mol else smiles_str
    except Exception:
        return smiles_str
    
def group_split(df, SMILES__1, SMILES__2, seed=SEED, val_size=0.2):
    """
    1. Holds out solvent mixture points for the test set.
    2. Uses GroupShuffleSplit on remaining data to prevent train/val mixture leakage.
    """
    target_1 = get_canonical_smiles(SMILES__1)
    target_2 = get_canonical_smiles(SMILES__2)

    s1_canon = df["SMILES_1"].apply(get_canonical_smiles)
    s2_canon = df["SMILES_2"].apply(get_canonical_smiles)

    # Identify holdout test pair (Water + 2-Propanol, excluding pure solvents)
    is_pair_1 = (s1_canon == target_1) & (s2_canon == target_2)
    is_pair_2 = (s1_canon == target_2) & (s2_canon == target_1)

    mask_test_pair = is_pair_1 | is_pair_2
    mask_pure_solvent = (df["comp_1"] == 0) | (df["comp_1"] == 100) | (df["comp_1"] == 1.0)
    mask_solv_test = mask_test_pair & (~mask_pure_solvent)

    # Split dataset into test and remaining subsets
    df_test = df[mask_solv_test]
    df_remaining = df[~mask_solv_test]

    idx_test = df_test.index.values

    # Create symmetric mixture group keys for df_remaining to prevent train/val leakage
    # Sorting SMILES pair handles order variations (e.g., A+B vs B+A)
    smiles_sorted = np.sort(df_remaining[["SMILES_1", "SMILES_2"]].values, axis=1)
    groups = (
        smiles_sorted[:, 0] + "|" + 
        smiles_sorted[:, 1] + "|" + 
        df_remaining["comp_1"].astype(str)
    )

    # GroupShuffleSplit for Train/Val
    gss = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx_rel, val_idx_rel = next(gss.split(df_remaining, groups=groups))

    # Map relative position indices back to original dataframe index labels
    idx_train = df_remaining.index[train_idx_rel].values
    idx_val = df_remaining.index[val_idx_rel].values

    return idx_train, idx_val, idx_test

def grouped_condition_split(
    df,
    seed=SEED,
    group_columns=("Temperature (K)", "comp_1", "comp_2"),  # <- adjust to your columns
    test_frac=0.15,
    val_frac=0.15,
):
    """
    Group-level split (fractions are relative to the full dataset):
      - test  = randomly chosen (temperature, composition) groups, ~15% of rows
      - val   = randomly chosen groups from the rest, ~15% of all rows
      - train = everything else, ~70% of all rows
    """
    rng = np.random.default_rng(seed)

    group_keys = (
        df[list(group_columns)].astype(str).agg("|".join, axis=1).to_numpy()
    )

    def pick_groups(candidate_idx, n_target):
        """Randomly pick whole groups until ~n_target rows are reached."""
        keys = group_keys[candidate_idx]
        unique_groups, inverse = np.unique(keys, return_inverse=True)
        counts = np.bincount(inverse)

        order = rng.permutation(len(unique_groups))

        chosen, total = [], 0
        for g in order:
            if total >= n_target:
                break
            chosen.append(g)
            total += counts[g]

        mask = np.isin(inverse, chosen)
        return candidate_idx[mask], candidate_idx[~mask]

    n = len(df)
    all_indices = np.arange(n)

    # 1) ~15% of all rows -> test
    idx_test, idx_rest = pick_groups(all_indices, test_frac * n)

    # 2) ~15% of all rows -> val, taken from the remainder
    idx_val, idx_train = pick_groups(idx_rest, val_frac * n)

    idx_train, idx_val, idx_test = (np.sort(i) for i in (idx_train, idx_val, idx_test))

    signature = hashlib.md5(
        np.concatenate([idx_train, idx_val, idx_test]).tobytes()
    ).hexdigest()

    return idx_train, idx_val, idx_test, signature, group_keys


def temperature_holdout_split(df, mode="extrapolation"):
    if mode not in {"interpolation", "extrapolation"}:
        raise ValueError("mode must be 'interpolation' or 'extrapolation'")

    group_columns = ["SMILES_1", "SMILES_2", "comp_1", "comp_2"]
    group_keys = df[group_columns].astype(str).agg("|".join, axis=1).to_numpy()
    temperatures = df["Temperature (K)"].to_numpy(float)
    train, val, test = [], [], []

    for group in np.unique(group_keys):
        indices = np.flatnonzero(group_keys == group)
        order = indices[np.argsort(temperatures[indices], kind="stable")]
        n = len(order)
        if n < 6:
            train.extend(order.tolist())
            continue

        if mode == "extrapolation":
            test_positions = {0, n - 1}
            val_positions = {1, n - 2}
        else:
            test_positions = {n // 2}
            val_positions = {n // 2 - 1}

        for position, index in enumerate(order):
            if position in test_positions:
                test.append(int(index))
            elif position in val_positions:
                val.append(int(index))
            else:
                train.append(int(index))

    idx_train = np.sort(np.asarray(train, dtype=int))
    idx_val = np.sort(np.asarray(val, dtype=int))
    idx_test = np.sort(np.asarray(test, dtype=int))
    signature = hashlib.md5(
        np.concatenate([idx_train, idx_val, idx_test]).tobytes()
    ).hexdigest()
    return idx_train, idx_val, idx_test, signature, group_keys


def load_raw_data(data_path):
    data_path = Path(data_path)
    df = pd.read_csv(data_path)
    required = {
        "LogS",
        "Temperature (K)",
        "SMILES_1",
        "SMILES_2",
        "comp_1",
        "comp_2",
    }
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    numeric = df[["LogS", "Temperature (K)", "comp_1", "comp_2"]].to_numpy(float)
    if not np.isfinite(numeric).all():
        raise ValueError("LogS, temperature, and composition must be finite")
    if (numeric[:, 1] <= 0).any():
        raise ValueError("Temperature (K) must be positive")
    total = numeric[:, 2] + numeric[:, 3]
    if (total <= 0).any():
        raise ValueError("Composition totals must be positive")
    x1 = (numeric[:, 2] / total).reshape(-1, 1)
    if ((x1 < 0) | (x1 > 1)).any():
        raise ValueError("Normalized composition must lie in [0, 1]")
    return df, x1, numeric[:, 1:2], numeric[:, 0:1]


def compute_fingerprints(df, calculator):
    """Compute each unique SMILES once instead of once per data row."""
    unique_smiles = pd.unique(pd.concat([df["SMILES_1"], df["SMILES_2"]]))
    lookup = {
        smiles: smiles_to_mordred(smiles, calculator) for smiles in unique_smiles
    }
    fp1 = np.vstack([lookup[s] for s in df["SMILES_1"]])
    fp2 = np.vstack([lookup[s] for s in df["SMILES_2"]])
    return fp1, fp2, lookup


def make_loaders(arrays_train, arrays_val, arrays_test, batch_size=32):
    train_t = [torch.tensor(a, dtype=torch.float32) for a in arrays_train]
    val_t = [torch.tensor(a, dtype=torch.float32) for a in arrays_val]
    test_t = [torch.tensor(a, dtype=torch.float32) for a in arrays_test]
    return (
        DataLoader(TensorDataset(*train_t), batch_size=batch_size, shuffle=True),
        DataLoader(TensorDataset(*val_t), batch_size=len(val_t[0]), shuffle=False),
        DataLoader(TensorDataset(*test_t), batch_size=len(test_t[0]), shuffle=False),
    )


ACTIVATIONS = {"Tanh": nn.Tanh, "ReLU": nn.ReLU, "Sigmoid": nn.Sigmoid, "ELU": nn.ELU}


def initialize_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        nn.init.constant_(module.bias, 0.01)


def generate_mlp(n_input, hidden_layers, n_out, dropout, activation):
    """Dropout is used only between hidden layers, never on model outputs."""
    sizes = [n_input] + list(hidden_layers) + [n_out]
    layers = []
    for index, (left, right) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(left, right))
        if index < len(sizes) - 2:
            layers.extend([ACTIVATIONS[activation](), nn.Dropout(dropout)])
    return nn.Sequential(*layers)


class WeightedFingerprintModel(nn.Module):
    def __init__(self, n_desc):
        super().__init__()
        # inputs: [fp1*x1, fp2*x2, T]
        self.mlp = generate_mlp(n_desc * 2 + 1, [64, 32], 1, 0.1, "Tanh")
        self.apply(initialize_weights)

    def forward(self, fp1, fp2, x1, temperature):
        x2 = 1.0 - x1
        inp = torch.cat([fp1 * x1, fp2 * x2, temperature.reshape(-1, 1)], dim=1)
        return self.mlp(inp)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class ConcatenatedCompositionModel(nn.Module):
    """Concatenate compact solvent descriptors with the explicit x1 fraction and T."""

    def __init__(self, n_desc):
        super().__init__()
        # inputs: [fp1, fp2, x1, T]
        self.mlp = generate_mlp(n_desc * 2 + 2, [64, 32], 1, 0.1, "Tanh")
        self.apply(initialize_weights)

    def forward(self, fp1, fp2, x1, temperature):
        inp = torch.cat([fp1, fp2, x1, temperature.reshape(-1, 1)], dim=1)
        return self.mlp(inp)

    def count_parameters(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)


class WeightedSumFingerprintModel(nn.Module):
    """Mix compact solvent descriptors directly before the neural backbone."""

    def __init__(self, n_desc):
        super().__init__()
        # inputs: [x1*fp1 + x2*fp2, T]
        self.mlp = generate_mlp(n_desc + 1, [64, 32], 1, 0.1, "Tanh")
        self.apply(initialize_weights)

    def forward(self, fp1, fp2, x1, temperature):
        x2 = 1.0 - x1
        mixed_descriptors = x1 * fp1 + x2 * fp2
        inp = torch.cat([mixed_descriptors, temperature.reshape(-1, 1)], dim=1)
        return self.mlp(inp)
    
    def count_parameters(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)


class PureComponentMixingModel(nn.Module):
    """Transform each pure solvent, mix the transformed spaces, then predict LogS."""

    def __init__(self, n_desc):
        super().__init__()
        self.pure_component_mlp = generate_mlp(n_desc, [64, 32], 16, 0.1, "Tanh")
        # inputs: [mixed latent (16), T]
        self.regressor = generate_mlp(16 + 1, [16], 1, 0.1, "Tanh")
        self.apply(initialize_weights)

    def forward(self, fp1, fp2, x1, temperature):
        x2 = 1.0 - x1
        z1 = self.pure_component_mlp(fp1)
        z2 = self.pure_component_mlp(fp2)
        mixed = x1 * z1 + x2 * z2
        return self.regressor(torch.cat([mixed, temperature.reshape(-1, 1)], dim=1))

    def count_parameters(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)


class WeightedEmbeddingModel(nn.Module):
    """Encode each solvent, then mix the latent solvent embeddings by composition."""

    def __init__(self, n_desc):
        super().__init__()
        self.solvent_encoder = generate_mlp(n_desc, [64, 32], 16, 0.1, "Tanh")
        # inputs: [mixed embedding (16), T]
        self.regressor = generate_mlp(16 + 1, [16], 1, 0.1, "Tanh")
        self.apply(initialize_weights)

    def forward(self, fp1, fp2, x1, temperature):
        x2 = 1.0 - x1
        emb1 = self.solvent_encoder(fp1)
        emb2 = self.solvent_encoder(fp2)
        mixed_embedding = x1 * emb1 + x2 * emb2
        return self.regressor(torch.cat([mixed_embedding, temperature.reshape(-1, 1)], dim=1))

    def count_parameters(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)

class OrderInvariantConcatModel(nn.Module):
    """Shared solvent encoder + composition weighting + order-averaged LogS head."""

    def __init__(self, n_desc):
        super().__init__()
        # Shared encoder: same weights used for solvent 1 and solvent 2.
        self.solvent_encoder = generate_mlp(n_desc, [64, 32], 16, 0.1, "Tanh")
        # Head sees [weighted emb 1 (16), weighted emb 2 (16), T (1)] -> LogS
        self.head = generate_mlp(16 + 16 + 1, [16], 1, 0.1, "Tanh")
        self.apply(initialize_weights)

    def forward(self, fp1, fp2, x1, temperature):
        x2 = 1.0 - x1
        t = temperature.reshape(-1, 1)  # already standardized in main()

        # Encode each solvent once (shared weights), then weight by mole fraction.
        e1 = self.solvent_encoder(fp1) * x1
        e2 = self.solvent_encoder(fp2) * x2

        # Same head, both concatenation orders -> removes order bias.
        out_12 = self.head(torch.cat([e1, e2, t], dim=1))
        out_21 = self.head(torch.cat([e2, e1, t], dim=1))

        # Average the final LogS predictions.
        return 0.5 * (out_12 + out_21)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_model(model, train_loader, val_loader, forward_batch, max_epochs=600, patience=50):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    criterion = nn.MSELoss()
    best_loss, best_state, wait = float("inf"), None, 0
    train_losses, val_losses = [], []
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = [x.to(DEVICE) for x in batch]
            optimizer.zero_grad()
            loss = criterion(forward_batch(model, batch), batch[-1])
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch[-1])
        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        with torch.no_grad():
            batch = [x.to(DEVICE) for x in next(iter(val_loader))]
            val_loss = criterion(forward_batch(model, batch), batch[-1]).item()
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if epoch % 50 == 0:
            print(f"Epoch {epoch:4d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
    if best_state is None:
        raise RuntimeError("No valid validation loss was observed")
    model.load_state_dict(best_state)
    return best_loss, train_losses, val_losses, epoch


def predict_model(model, loader, forward_batch):
    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = [x.to(DEVICE) for x in batch]
            predictions.append(forward_batch(model, batch).detach().cpu().numpy().ravel())
            targets.append(batch[-1].detach().cpu().numpy().ravel())
    return np.concatenate(targets), np.concatenate(predictions)


def metrics(y_true, y_pred):
    residual = y_true - y_pred
    denominator = np.sum((y_true - y_true.mean()) ** 2)
    r2 = np.nan if denominator == 0 else 1.0 - np.sum(residual**2) / denominator
    return {
        "r2": float(r2),
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "mae": float(np.mean(np.abs(residual))),
    }

def plot_parity(results, save_path, title="Parity plot"):
    """
    results: dict like {"Train": (y_true, y_pred), "Val": (...), "Test": (...)}
    All values must be in original LogS units.
    """
    colors = {"Train": "tab:blue", "Val": "tab:orange", "Test": "tab:green"}

    all_vals = np.concatenate([np.concatenate(v) for v in results.values()])
    lo, hi = all_vals.min(), all_vals.max()
    pad = 0.05 * (hi - lo)
    lims = (lo - pad, hi + pad)

    fig, axes = plt.subplots(1, len(results) + 1, figsize=(5 * (len(results) + 1), 5))

    # Combined panel
    ax = axes[0]
    for name, (yt, yp) in results.items():
        m = metrics(yt, yp)
        ax.scatter(yt, yp, s=40, alpha=0.5, color=colors.get(name), label=f"{name}")
    ax.plot(lims, lims, "k--", lw=1)
    ax.set(xlim=lims, ylim=lims)
    ax.set_xlabel("Experimental LogS", fontsize=18)
    ax.set_ylabel("Predicted LogS", fontsize=18)
    ax.set_aspect("equal")
    ax.legend()

    # One panel per split
    for ax, (name, (yt, yp)) in zip(axes[1:], results.items()):
        m = metrics(yt, yp)
        ax.scatter(yt, yp, s=12, alpha=0.5, color=colors.get(name))
        ax.plot(lims, lims, "k--", lw=1)
        ax.set(xlim=lims, ylim=lims, xlabel="Experimental LogS", ylabel="Predicted LogS",
               title=f"{name} (n={len(yt)})")
        ax.set_aspect("equal")
        ax.text(0.05, 0.95,
                f"R² = {m['r2']:.3f}\nRMSE = {m['rmse']:.3f}\nMAE = {m['mae']:.3f}",
                transform=ax.transAxes, va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)



In [8]:
# ============================================================================
# NO PCA - NO pretrain - NO Physics
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from rdkit import Chem

def main(data_path, output_dir):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load data and run custom split
    df, x1, temperature, target = load_raw_data(data_path)
    idx_train, idx_val, idx_test, sig, keys = grouped_condition_split(df, seed=SEED)

    # 2. Compute full Mordred descriptors (No PCA)
    calculator = make_calculator()
    fp1_raw, fp2_raw, descriptor_lookup = compute_fingerprints(df, calculator)
    
    # 3. Fit standardizer strictly on training data
    fp_preprocessor = TrainOnlyStandardizer().fit(
        [fp1_raw[idx_train], fp2_raw[idx_train]]
    )
    fp1 = fp_preprocessor.transform(fp1_raw)
    fp2 = fp_preprocessor.transform(fp2_raw)

    # 4. Split data into PyTorch loaders
    split = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_train, fp1_val, fp1_test = split(fp1)
    fp2_train, fp2_val, fp2_test = split(fp2)
    x_train, x_val, x_test = split(x1)
    t_train, t_val, t_test = split(temperature)
    y_train, y_val, y_test = split(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_train, fp2_train, x_train, t_train, y_train],
        [fp1_val, fp2_val, x_val, t_val, y_val],
        [fp1_test, fp2_test, x_test, t_test, y_test],
    )

    # 5. Model initialization & training
    model = WeightedFingerprintModel(n_desc=fp1.shape[1]).to(DEVICE)
    print(f"Total Rows: {len(df)} | test rows: {len(idx_test)}")
    print(f"Split sizes: train={len(idx_train)} val={len(idx_val)} test={len(idx_test)}")
    print(f"Total descriptor features: {fp1_raw.shape[1]}")
    print(f"Retained descriptor features: {fp1.shape[1]}")
    
    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val, _, _, epochs = train_model(model, train_loader, val_loader, forward)
    print(f"\nBest validation loss: {best_val:.6f} (last epoch {epochs})")

    # 6. Predict and calculate metrics for all splits
    y_true_train, y_pred_train = predict_model(model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")
    
    plot_parity(
    {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
    output_dir / "Concatenated.png",
    title="Clozapine LogS Prediction Parity Plot"
)

parser = argparse.ArgumentParser()
parser.add_argument("--data", default="Clozapine_dataset.csv")
parser.add_argument("--output-dir", default="Baseline_outputs")
args = parser.parse_args([])


main(args.data, args.output_dir)

[TrainOnlyStandardizer] Descriptors: 681 total | 186 dropped (NaN) | 296 dropped (constant) | 199 remaining
Total Rows: 684 | test rows: 103
Split sizes: train=478 val=103 test=103
Total descriptor features: 681
Retained descriptor features: 199
Epoch   50 | Train: 0.0769 | Val: 0.0338
Epoch  100 | Train: 0.0565 | Val: 0.0256
Epoch  150 | Train: 0.0406 | Val: 0.0225
Epoch  200 | Train: 0.0353 | Val: 0.0367
Epoch  250 | Train: 0.0361 | Val: 0.0245
Epoch  300 | Train: 0.0331 | Val: 0.0193
Epoch  350 | Train: 0.0392 | Val: 0.0191
Epoch  400 | Train: 0.0301 | Val: 0.0140
Epoch  450 | Train: 0.0263 | Val: 0.0139
Epoch  500 | Train: 0.0308 | Val: 0.0083

Best validation loss: 0.007564 (last epoch 521)
Train | R²=0.9907 RMSE=0.0878 MAE=0.0531
Val   | R²=0.9918 RMSE=0.0870 MAE=0.0528
Test  | R²=0.9882 RMSE=0.1038 MAE=0.0612


In [3]:
# ============================================================================
# NO PCA - NO pretrain - NO Physics
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from rdkit import Chem

def main(data_path, output_dir):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load data and run custom split
    df, x1, temperature, target = load_raw_data(data_path)
    idx_train, idx_val, idx_test, sig, keys = grouped_condition_split(df, seed=SEED)

    # 2. Compute full Mordred descriptors (No PCA)
    calculator = make_calculator()
    fp1_raw, fp2_raw, descriptor_lookup = compute_fingerprints(df, calculator)
    
    # 3. Fit standardizer strictly on training data
    fp_preprocessor = TrainOnlyStandardizer().fit(
        [fp1_raw[idx_train], fp2_raw[idx_train]]
    )
    fp1 = fp_preprocessor.transform(fp1_raw)
    fp2 = fp_preprocessor.transform(fp2_raw)

    # 4. Split data into PyTorch loaders
    split = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_train, fp1_val, fp1_test = split(fp1)
    fp2_train, fp2_val, fp2_test = split(fp2)
    x_train, x_val, x_test = split(x1)
    t_train, t_val, t_test = split(temperature)
    y_train, y_val, y_test = split(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_train, fp2_train, x_train, t_train, y_train],
        [fp1_val, fp2_val, x_val, t_val, y_val],
        [fp1_test, fp2_test, x_test, t_test, y_test],
    )

    # 5. Model initialization & training
    model = WeightedSumFingerprintModel(n_desc=fp1.shape[1]).to(DEVICE)
    print(f"Total Rows: {len(df)} | test rows: {len(idx_test)}")
    print(f"Split sizes: train={len(idx_train)} val={len(idx_val)} test={len(idx_test)}")
    print(f"Total descriptor features: {fp1_raw.shape[1]}")
    print(f"Retained descriptor features: {fp1.shape[1]}")
    
    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val, _, _, epochs = train_model(model, train_loader, val_loader, forward)
    print(f"\nBest validation loss: {best_val:.6f} (last epoch {epochs})")

    # 6. Predict and calculate metrics for all splits
    y_true_train, y_pred_train = predict_model(model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")
    
    plot_parity(
    {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
    output_dir / "Sumweighted.png",
    title="Clozapine LogS Prediction Parity Plot"
)

parser = argparse.ArgumentParser()
parser.add_argument("--data", default="Clozapine_dataset.csv")
parser.add_argument("--output-dir", default="Baseline_outputs")
args = parser.parse_args([])


main(args.data, args.output_dir)

[TrainOnlyStandardizer] Descriptors: 681 total | 186 dropped (NaN) | 296 dropped (constant) | 199 remaining
Total Rows: 684 | test rows: 103
Split sizes: train=478 val=103 test=103
Total descriptor features: 681
Retained descriptor features: 199
Epoch   50 | Train: 0.0783 | Val: 0.0468
Epoch  100 | Train: 0.0612 | Val: 0.0426
Epoch  150 | Train: 0.0552 | Val: 0.0363
Epoch  200 | Train: 0.0548 | Val: 0.0418
Epoch  250 | Train: 0.0560 | Val: 0.0347

Best validation loss: 0.028685 (last epoch 263)
Train | R²=0.9764 RMSE=0.1399 MAE=0.0960
Val   | R²=0.9690 RMSE=0.1694 MAE=0.1098
Test  | R²=0.9686 RMSE=0.1692 MAE=0.1161


In [4]:
# ============================================================================
# NO PCA - NO pretrain - NO Physics
# ============================================================================
import argparse
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from rdkit import Chem

def main(data_path, output_dir):
    seed_everything(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load data and run custom split
    df, x1, temperature, target = load_raw_data(data_path)
    idx_train, idx_val, idx_test, sig, keys = grouped_condition_split(df, seed=SEED)

    # 2. Compute full Mordred descriptors (No PCA)
    calculator = make_calculator()
    fp1_raw, fp2_raw, descriptor_lookup = compute_fingerprints(df, calculator)
    
    # 3. Fit standardizer strictly on training data
    fp_preprocessor = TrainOnlyStandardizer().fit(
        [fp1_raw[idx_train], fp2_raw[idx_train]]
    )
    fp1 = fp_preprocessor.transform(fp1_raw)
    fp2 = fp_preprocessor.transform(fp2_raw)

    # 4. Split data into PyTorch loaders
    split = lambda a: (a[idx_train], a[idx_val], a[idx_test])
    fp1_train, fp1_val, fp1_test = split(fp1)
    fp2_train, fp2_val, fp2_test = split(fp2)
    x_train, x_val, x_test = split(x1)
    t_train, t_val, t_test = split(temperature)
    y_train, y_val, y_test = split(target)

    train_loader, val_loader, test_loader = make_loaders(
        [fp1_train, fp2_train, x_train, t_train, y_train],
        [fp1_val, fp2_val, x_val, t_val, y_val],
        [fp1_test, fp2_test, x_test, t_test, y_test],
    )

    # 5. Model initialization & training
    model = OrderInvariantConcatModel(n_desc=fp1.shape[1]).to(DEVICE)
    print(f"Total Rows: {len(df)} | test rows: {len(idx_test)}")
    print(f"Split sizes: train={len(idx_train)} val={len(idx_val)} test={len(idx_test)}")
    print(f"Total descriptor features: {fp1_raw.shape[1]}")
    print(f"Retained descriptor features: {fp1.shape[1]}")
    
    forward = lambda m, batch: m(batch[0], batch[1], batch[2], batch[3])
    best_val, _, _, epochs = train_model(model, train_loader, val_loader, forward)
    print(f"\nBest validation loss: {best_val:.6f} (last epoch {epochs})")

    # 6. Predict and calculate metrics for all splits
    y_true_train, y_pred_train = predict_model(model, train_loader, forward)
    y_true_val, y_pred_val = predict_model(model, val_loader, forward)
    y_true_test, y_pred_test = predict_model(model, test_loader, forward)

    m_train = metrics(y_true_train, y_pred_train)
    m_val = metrics(y_true_val, y_pred_val)
    m_test = metrics(y_true_test, y_pred_test)

    print(f"Train | R²={m_train['r2']:.4f} RMSE={m_train['rmse']:.4f} MAE={m_train['mae']:.4f}")
    print(f"Val   | R²={m_val['r2']:.4f} RMSE={m_val['rmse']:.4f} MAE={m_val['mae']:.4f}")
    print(f"Test  | R²={m_test['r2']:.4f} RMSE={m_test['rmse']:.4f} MAE={m_test['mae']:.4f}")
    
    plot_parity(
    {"Train": (y_true_train, y_pred_train), "Val": (y_true_val, y_pred_val), "Test": (y_true_test, y_pred_test)},
    output_dir / "OrderInvariantConcat.png",
    title="Clozapine LogS Prediction Parity Plot"
)

parser = argparse.ArgumentParser()
parser.add_argument("--data", default="Clozapine_dataset.csv")
parser.add_argument("--output-dir", default="Baseline_outputs")
args = parser.parse_args([])


main(args.data, args.output_dir)

[TrainOnlyStandardizer] Descriptors: 681 total | 186 dropped (NaN) | 296 dropped (constant) | 199 remaining
Total Rows: 684 | test rows: 103
Split sizes: train=478 val=103 test=103
Total descriptor features: 681
Retained descriptor features: 199
Epoch   50 | Train: 0.1634 | Val: 0.1014
Epoch  100 | Train: 0.1371 | Val: 0.0844
Epoch  150 | Train: 0.1323 | Val: 0.0852
Epoch  200 | Train: 0.1624 | Val: 0.0885
Epoch  250 | Train: 0.1252 | Val: 0.0736
Epoch  300 | Train: 0.1155 | Val: 0.0734
Epoch  350 | Train: 0.1176 | Val: 0.0728
Epoch  400 | Train: 0.1317 | Val: 0.0742

Best validation loss: 0.065443 (last epoch 428)
Train | R²=0.9423 RMSE=0.2190 MAE=0.1492
Val   | R²=0.9292 RMSE=0.2558 MAE=0.1641
Test  | R²=0.9311 RMSE=0.2507 MAE=0.1634
